In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import sys
from pathlib import Path
import os
import flap

sys.path.insert(0, str(Path.cwd().parent))
from neuro_bes import data

In [ ]:
shot='20250513.032'
path='/data2/W7-X/processed_data/APDCAM/flap_recon/'
try:
    file_list=os.listdir(os.path.join(path,shot))
except FileNotFoundError:
    raise ValueError("Directory " + os.path.join(path,shot) + " does not exist")

file_name_light=[i for i in file_list if ('light_orig' in i)]
file_name_light_ds=[i for i in file_list if ('light_ds_orig' in i)]
file_name_density=[i for i in file_list if ('dens' in i)]
file_name_light_recon=[i for i in file_list if ('light_recon' in i)]
if len(file_name_density)>1:
    raise ValueError("There is more than one data source for shot " + shot)
if len(file_name_density)==0:
    raise ValueError("Data source not found for shot " + shot)
file_name_light=file_name_light[0]
file_name_light_ds=file_name_light_ds[0]
file_name_density=file_name_density[0]
file_name_light_recon=file_name_light_recon[0]

In [ ]:
light=flap.load(os.path.join(os.path.join(path,shot),file_name_light))
light_ds=flap.load(os.path.join(os.path.join(path,shot),file_name_light_ds))
density=flap.load(os.path.join(os.path.join(path,shot),file_name_density))
light_recon=flap.load(os.path.join(os.path.join(path,shot),file_name_light_recon))

In [ ]:
r_coord=light.coordinate('Device R')[0][0]
time_instances_density=density.coordinate('Time')[0][:,0]
time_instances_light=light.coordinate('Time')[0][:,0]

In [ ]:
density_data=density.data
light_data=light.data
light_ds_data=light_ds.data
light_recon_data=light_recon.data

In [ ]:
missing = np.setdiff1d(time_instances_light, time_instances_density, assume_unique=True)
if missing.size:
    all_times = np.sort(np.concatenate([time_instances_density, missing]))
    new_density = np.full((all_times.shape[0],) + density.data.shape[1:], np.nan, dtype=density.data.dtype)
    idx = np.searchsorted(all_times, time_instances_density)
    new_density[idx] = density.data
    density_data = new_density
    time_instances_density = all_times
        

In [ ]:
plt.figure(figsize=(15,3))
plt.subplot(1,3,1)
# colorbar with same scale for both plots
im = plt.pcolormesh(time_instances_density[::10000], r_coord, density_data[::10000].T, cmap='coolwarm')
plt.subplot(1,3,2)
# colorbar with same scale for both plots
im = plt.pcolormesh(time_instances_light[::10], r_coord, light_data[::10].T, cmap='coolwarm')

In [ ]:
plt.figure(figsize=(12,4))
plt.subplot(1,4,1)
plt.plot(r_coord, light_data[::1000].T)
plt.subplot(1,4,2)
plt.plot(r_coord, light_ds_data.T) 
plt.subplot(1,4,3)
plt.plot(r_coord, light_recon_data.T) 
plt.subplot(1,4,4)
plt.plot(r_coord, density_data[::10000].T)

In [ ]:
grid=r_coord
energy=0
species="Na"
ID="we_"+shot
verbose="W7X experimental data shot "+shot+", full shot, no curation"
zeff=0
q=0
temperature=np.array(0)
tags=['Time instance ' + str(i) + ' s' for i in time_instances_density]

In [ ]:
grid

In [ ]:
test_data=data.besInferenceDatapoints(grid=grid,energy=energy,species=species,ID=ID,zeff=zeff,q=q,temperature=temperature,verbose=verbose)

In [ ]:
test_data.add_datapoints_bulk(density_data, light_data, tags)

In [ ]:
density_data.shape

In [ ]:
plt.plot(test_data.grid,test_data.emissions[10])
plt.plot(test_data.grid,light_recon_data[10])

In [ ]:
#test_data.export_to_h5(path_to_dir="/home/molnarbalazs/data/BES_ML_modelling")